In [ ]:
#Pandas import and folder path
import pandas as pd
deirectoryPath ="../data/"

In [ ]:
#Reading dataset
dataDF = pd.read_csv(f"{deirectoryPath}NF-UNSW-NB15-v2.csv")

In [ ]:
dataDF.head()

In [ ]:
len(dataDF.columns)

In [ ]:
len(dataDF['IPV4_SRC_ADDR'].unique())

In [ ]:
len(dataDF['IPV4_DST_ADDR'].unique())

In [ ]:
#Droping unwanted features
dataDF.drop(columns=["Attack" , "L4_SRC_PORT"] , inplace = True , axis = 1)
dataDF.head()

In [ ]:
#Train test split(Based on edges)
from sklearn.model_selection import train_test_split

trainDF , testDF = train_test_split(dataDF , test_size=0.25 , random_state=20 , stratify=dataDF['Label'])

In [ ]:
#Initializing train and test graph
import networkx as netx
trainingGraph = netx.MultiDiGraph()
testingGraph = netx.MultiDiGraph()

In [ ]:
dataDF.columns

In [ ]:
#Defining node based on IP
srcTrain = trainDF['IPV4_SRC_ADDR'].to_list()
destTrain = trainDF['IPV4_DST_ADDR'].to_list() 
srcTest = testDF['IPV4_SRC_ADDR'].to_list()
destTest = testDF['IPV4_DST_ADDR'].to_list() 

In [ ]:
#Adding node to graphs
trainingGraph.add_nodes_from(srcTrain)
trainingGraph.add_nodes_from(destTrain)
testingGraph.add_nodes_from(srcTest)
testingGraph.add_nodes_from(destTest)

In [ ]:
#Preparing col for features
feature_cols = [col for col in trainDF.columns if col not in ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label']]

In [ ]:
#Splitting continuous values and categorical values and standarizing continuous values
from sklearn.preprocessing import StandardScaler
scalerEdge = StandardScaler()
train_feats = scalerEdge.fit_transform(trainDF[feature_cols].values.astype(float))
scalerEdge = StandardScaler()

cat_cols = [
    'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
    'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS',
    'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_TYPE',
    'FTP_COMMAND_RET_CODE'
]
cont_cols = [col for col in feature_cols if col not in cat_cols]

train_cont = trainDF[cont_cols].astype(float)
test_cont = testDF[cont_cols].astype(float)

train_feats_cont = pd.DataFrame(
    scalerEdge.fit_transform(train_cont),
    columns=cont_cols,
    index=trainDF.index
)
test_feats_cont = pd.DataFrame(
    scalerEdge.transform(test_cont),
    columns=cont_cols,
    index=testDF.index
)

train_feats = pd.concat(
    [train_feats_cont, trainDF[cat_cols].astype(float)],
    axis=1
)[feature_cols].to_numpy()

test_feats = pd.concat(
    [test_feats_cont, testDF[cat_cols].astype(float)],
    axis=1
)[feature_cols].to_numpy()

In [ ]:
for i, (idx, row) in enumerate(trainDF.iterrows()):
    trainingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=train_feats[i], label=row['Label'])

In [ ]:
for i, (idx, row) in enumerate(testDF.iterrows()):
    testingGraph.add_edge(row['IPV4_SRC_ADDR'], row['IPV4_DST_ADDR'],
                            features=test_feats[i], label=row['Label'])

In [ ]:
print(f"Training Graph - Nodes: {trainingGraph.number_of_nodes()}, Edges: {trainingGraph.number_of_edges()}")
print(f"Test Graph - Nodes: {testingGraph.number_of_nodes()}, Edges: {testingGraph.number_of_edges()}")
print(f"Edge features shape: {trainingGraph[list(trainingGraph.edges())[0][0]][list(trainingGraph.edges())[0][1]][0]['features'].shape}")

In [ ]:
src_stats_train = trainDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_train = trainDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [ ]:
src_stats_test = testDF.groupby("IPV4_SRC_ADDR").agg(
    out_degree=("IPV4_SRC_ADDR", "count"),
    avg_sbytes=("IN_BYTES", "mean"),
    avg_spkts=("IN_PKTS", "mean")
)

dst_stats_test = testDF.groupby("IPV4_DST_ADDR").agg(
    in_degree=("IPV4_DST_ADDR", "count"),
    avg_dbytes=("OUT_BYTES", "mean"),
    avg_dpkts=("OUT_PKTS", "mean")
)

In [ ]:
nodeOrder = list(trainingGraph.nodes())
trainNodeFeature = pd.concat(
    [src_stats_train, dst_stats_train],
    axis=1
)

trainNodeFeature = trainNodeFeature.reindex(nodeOrder).fillna(0)
assert list(trainNodeFeature.index) == nodeOrder

In [ ]:
nodeOrder = list(testingGraph.nodes())
testNodeFeature = pd.concat(
    [src_stats_test, dst_stats_test],
    axis=1
)

testNodeFeature = testNodeFeature.reindex(nodeOrder).fillna(0)
assert list(testNodeFeature.index) == nodeOrder

In [ ]:
nodeScaler = StandardScaler()
trainNodeFeature[:] = nodeScaler.fit_transform(trainNodeFeature)

In [ ]:
testNodeFeature[:] = nodeScaler.transform(testNodeFeature)

In [ ]:
import torch
from torch_geometric.utils import from_networkx

trainGraphData = from_networkx(trainingGraph)
testGraphData = from_networkx(testingGraph)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
trainGraphData.x = torch.tensor(trainNodeFeature.values, dtype=torch.float).to(device)
testGraphData.x = torch.tensor(testNodeFeature.values, dtype=torch.float).to(device)

In [ ]:
edge_features_train = []
edge_labels_train = []

for _ , _ , attr in trainingGraph.edges(data=True):
    edge_features_train.append(attr['features'])
    edge_labels_train.append(attr['label'])

In [ ]:
edge_features_test = []
edge_labels_test = []

for _ , _ , attr in testingGraph.edges(data=True):
    edge_features_test.append(attr['features'])
    edge_labels_test.append(attr['label'])

In [ ]:
len(edge_features_train[0])

In [ ]:
trainGraphData.edge_attr = torch.tensor(
    edge_features_train,
    dtype=torch.float
).to(device)

trainGraphData.edge_label = torch.tensor(
    edge_labels_train,
    dtype=torch.float
).to(device)

In [ ]:
testGraphData.edge_attr = torch.tensor(
    edge_features_test,
    dtype=torch.float
).to(device)

testGraphData.edge_label = torch.tensor(
    edge_labels_test,
    dtype=torch.float
).to(device)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as funct
from torch_geometric.nn import GCNConv

In [ ]:
print(trainGraphData)

In [ ]:
# shapes must line up: edge_attr rows == edge_index columns
assert trainGraphData.edge_attr.shape[0] == trainGraphData.edge_index.shape[1]
assert testGraphData.edge_attr.shape[0] == testGraphData.edge_index.shape[1]

# node feature rows must match node count
assert trainGraphData.x.shape[0] == trainGraphData.num_nodes
assert testGraphData.x.shape[0] == testGraphData.num_nodes

In [ ]:
class EdgeGCN(nn.Module):
    def __init__(self, nodeFeatureDim , edgeFeatureDim , embeddingDim , labelDim):
        super().__init__()
        #GCN
        self.gcnConvoLayer1 = GCNConv(nodeFeatureDim , 64)
        self.gcnConvoLayer2 = GCNConv(64 , 32)
        self.gcnConvoLayer3 = GCNConv(32 , embeddingDim)

        #Classifier
        self.classifierLinearModel = nn.Sequential(
            nn.Linear(2*embeddingDim + edgeFeatureDim , 128),
            nn.ReLU(),
            nn.Linear(128 , 64),
            nn.ReLU(),
            nn.Linear(64 , 32),
            nn.ReLU(),
            nn.Linear(32,labelDim)
        )

    def forward(self, x, edge_index, edge_label_index, edge_attr):
        x = self.gcnConvoLayer1(x, edge_index)
        x = funct.relu(x)
        x = self.gcnConvoLayer2(x, edge_index)
        x = funct.relu(x)
        x = self.gcnConvoLayer3(x, edge_index)

        src = x[edge_label_index[0]]
        dst = x[edge_label_index[1]]

        edgeRepresentation = torch.cat([src, dst, edge_attr], dim=1)
        out = self.classifierLinearModel(edgeRepresentation)
        return out


In [ ]:
classifier = EdgeGCN(
    nodeFeatureDim=trainGraphData.x.shape[1],
    edgeFeatureDim=trainGraphData.edge_attr.shape[1],
    embeddingDim=16,
    labelDim=2
).to(device)

In [ ]:
label_counts = trainDF['Label'].value_counts().sort_index()
print(label_counts)

total = label_counts.sum()
num_classes = len(label_counts)

weight = total / (num_classes * label_counts)
weight = torch.tensor(weight.values, dtype=torch.float).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    classifier.parameters(),
    lr = 0.001
)
criterion = nn.CrossEntropyLoss(weight=weight)

In [ ]:
trainGraphData = trainGraphData.to(device)
testGraphData = testGraphData.to(device)

In [ ]:
from torch_geometric.loader import LinkNeighborLoader

train_loader = LinkNeighborLoader(
    data=trainGraphData,
    num_neighbors=[10, 10, 10],          # one entry per conv layer
    edge_label_index=trainGraphData.edge_index,
    edge_label=trainGraphData.edge_label,
    batch_size=2048,
    shuffle=True,
)

test_loader = LinkNeighborLoader(
    data=testGraphData,
    num_neighbors=[10, 10, 10],
    edge_label_index=testGraphData.edge_index,
    edge_label=testGraphData.edge_label,
    batch_size=2048,
    shuffle=False,                        # no need to shuffle eval
)

In [ ]:
epochs = 200

for epoch in range(epochs):
    classifier.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        edge_attr_batch = trainGraphData.edge_attr[batch.input_id].to(device)

        optimizer.zero_grad(set_to_none=True)
        out = classifier(
            batch.x,
            batch.edge_index,
            batch.edge_label_index,
            edge_attr_batch
        )
        loss = criterion(out, batch.edge_label.long())
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.edge_label.size(0)

    avg_loss = total_loss / trainGraphData.edge_index.size(1)
    print(f"Epoch {epoch+1:03d} Loss: {avg_loss:.4f}")